# Q-MAPPO Phase 2 — 500K Steps — Kaggle v3 (~5000 params)

**v3 improvements over v2:**
- N_QUBITS: 8 → **24** (3x more capacity)
- N_VQC_LAYERS: 5 → **6**
- Pre-encoder: 18→32→24 (wider)
- ENTROPY_COEF: 0.01 → **0.005**
- Total actor params: 1,357 → **5,413** (still 95.8% fewer than Phase 1)

**Phase 1 MAPPO 500K baseline:** Best Avg100=-40.81 | Eval Mean=-47.14
**Phase 2 v2 500K result:** Best Avg100=-52.85 | Eval Mean=-49.60 (actor was too small)

In [1]:
# CELL 1 — Install
!pip install -q pettingzoo==1.24.3 gymnasium==0.29.1 supersuit==3.9.3
print('Install complete')

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 847.8/847.8 kB 19.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 953.9/953.9 kB 45.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.2/50.2 kB 3.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 563.6/563.6 kB 31.2 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
kaggle-environments 1.29.3 requires gymnasium==1.2.0, but you have gymnasium 0.29.1 which is incompatible.
kaggle-environments 1.29.3 requires pettingzoo==1.24.0, but you have pettingzoo 1.24.3 which is incompatible.
shimmy 2.0.1 requires gymnasium>=1.0.0, but you have gymnasium 0.29.1 which is incompatible.
dopamine-rl 4.1.2 requires gym<=0.25.2, but you have gym 0.26.2 which is incompatible.
dopamine-rl 4.1.2 requires gymnasium>=1.0.0, but you have gymnasium 0.29.1 which is incompatible.
Install c

In [2]:
# CELL 2 — GPU Check + Save Directory
import torch, os, sys
print(f'Torch  : {torch.__version__}')
print(f'CUDA   : {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'Device : {torch.cuda.get_device_name(0)}')
    print(f'Memory : {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB')
else:
    print('WARNING: No GPU — Settings -> Accelerator -> GPU T4')

# Kaggle: save to /kaggle/working  (no Google Drive needed)
SAVE_DIR  = '/kaggle/working'
LOCAL_DIR = '/kaggle/working'
DRIVE_DIR = '/kaggle/working'   # same as LOCAL on Kaggle
os.makedirs(SAVE_DIR, exist_ok=True)
print(f'Save dir: {SAVE_DIR}')

Torch  : 2.10.0+cu128
CUDA   : True
Device : Tesla P100-PCIE-16GB
Memory : 17.1 GB
Save dir: /kaggle/working


/usr/local/lib/python3.12/dist-packages/torch/cuda/__init__.py:435: UserWarning: 
    Found GPU0 Tesla P100-PCIE-16GB which is of cuda capability 6.0.
    Minimum and Maximum cuda capability supported by this version of PyTorch is
    (7.0) - (12.0)
    
  queued_call()
/usr/local/lib/python3.12/dist-packages/torch/cuda/__init__.py:435: UserWarning: 
    Please install PyTorch with a following CUDA
    configurations:  12.6 following instructions at
    https://pytorch.org/get-started/locally/
    
  queued_call()
/usr/local/lib/python3.12/dist-packages/torch/cuda/__init__.py:435: UserWarning: 
Tesla P100-PCIE-16GB with CUDA capability sm_60 is not compatible with the current PyTorch installation.
The current PyTorch install supports CUDA capabilities sm_70 sm_75 sm_80 sm_86 sm_90 sm_100 sm_120.
If you want to use the Tesla P100-PCIE-16GB GPU with PyTorch, please check the instructions at https://pytorch.org/get-started/locally/

  queued_call()


In [3]:
# CELL 3 — Imports
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.distributions import Categorical
import matplotlib.pyplot as plt
import matplotlib
matplotlib.use('Agg')
import time, os, pickle, zipfile, shutil, csv
from datetime import datetime
from pettingzoo.mpe import simple_spread_v3
print('All imports done.')

All imports done.


/usr/local/lib/python3.12/dist-packages/pygame/pkgdata.py:25: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import resource_stream, resource_exists


In [4]:
# CELL 4 — Training Logger (saves CSV to /kaggle/working)
import csv
from datetime import datetime

class TrainingLogger:
    """Saves full training log as CSV to /kaggle/working (download from Output tab)."""
    def __init__(self, drive_dir, local_dir, run_name='qmappo_v3_500k'):
        self.log_path   = os.path.join(local_dir, f'{run_name}_log.csv')
        self.drive_path = self.log_path   # same path on Kaggle
        self.fields = [
            'episode','timestep','ep_reward','avg100','best_avg100',
            'actor_loss','critic_loss','entropy','actor_lr','elapsed_min'
        ]
        with open(self.log_path, 'w', newline='') as f:
            csv.DictWriter(f, fieldnames=self.fields).writeheader()
        print(f'Logger ready -> {self.log_path}')

    def log(self, row):
        with open(self.log_path, 'a', newline='') as f:
            csv.DictWriter(f, fieldnames=self.fields).writerow(row)

    def save_to_drive(self):
        # On Kaggle: file already in /kaggle/working — nothing to copy
        print(f'  [Saved] {datetime.now().strftime("%H:%M:%S")} -> {self.log_path}')

    def summary(self):
        import pandas as pd
        df = pd.read_csv(self.log_path)
        print(f'  Total rows logged : {len(df)}')
        print(f'  Best Avg100       : {df["best_avg100"].max():.2f}')
        print(f'  Log path          : {self.log_path}')

logger = TrainingLogger(SAVE_DIR, LOCAL_DIR, run_name='qmappo_v3_500k')
print('Logger created.')

Logger ready -> /kaggle/working/qmappo_v3_500k_log.csv
Logger created.


In [5]:
# CELL 5 — Environment
from pettingzoo.mpe import simple_spread_v3

env = simple_spread_v3.parallel_env(
    N=3, local_ratio=0.5, max_cycles=50, continuous_actions=False
)
obs, _ = env.reset()

AGENTS           = ['agent_0', 'agent_1', 'agent_2']
NUM_AGENTS       = len(AGENTS)
OBS_DIM          = env.observation_space(env.agents[0]).shape[0]  # 18
ACTION_DIM       = env.action_space(env.agents[0]).n               # 5
GLOBAL_STATE_DIM = OBS_DIM * NUM_AGENTS                            # 54

# v3: ~5000 params
N_QUBITS     = 24   # IMPROVED: 8->24 (3x more capacity)
N_VQC_LAYERS = 6    # IMPROVED: 5->6

print(f'Agents={NUM_AGENTS} | ObsDim={OBS_DIM} | Actions={ACTION_DIM} | GlobalDim={GLOBAL_STATE_DIM}')
print(f'QI Actor v3: {N_QUBITS} hidden units, {N_VQC_LAYERS} layers (~5,413 params)')

Agents=3 | ObsDim=18 | Actions=5 | GlobalDim=54
QI Actor v3: 24 hidden units, 6 layers (~5,413 params)


error: XDG_RUNTIME_DIR not set in the environment.


In [6]:
# CELL 5 — Running Normalizer (identical to Phase 1 MAPPO)
class RunningNorm:
    def __init__(self, shape, clip=5.0, eps=1e-8):
        self.shape = (shape,) if isinstance(shape, int) else shape
        self.mean  = np.zeros(self.shape, np.float64)
        self.var   = np.ones(self.shape,  np.float64)
        self.count = eps
        self.clip  = clip
        self.eps   = eps

    def update(self, x):
        x = np.asarray(x, np.float64).reshape(self.shape)
        self.count += 1
        delta       = x - self.mean
        self.mean  += delta / self.count
        self.var   += delta * (x - self.mean)

    @property
    def std(self):
        return np.sqrt(self.var / self.count + self.eps)

    def normalize(self, x):
        x_norm = (np.asarray(x, np.float64) - self.mean) / self.std
        return np.clip(x_norm, -self.clip, self.clip).astype(np.float32)

    def normalize_scalar(self, x):
        return float(np.clip(x / self.std[0], -self.clip, self.clip))

    def save(self):
        return {'mean': self.mean.copy(), 'var': self.var.copy(), 'count': self.count}

    def load(self, d):
        self.mean = d['mean']; self.var = d['var']; self.count = d['count']


obs_norm = RunningNorm(shape=OBS_DIM, clip=5.0)
ret_norm = RunningNorm(shape=1, clip=5.0)
print('Normalizers ready.')

Normalizers ready.


In [7]:
# CELL 6 — Orthogonal Init Helper (identical to Phase 1)
def ortho(layer, gain=np.sqrt(2)):
    nn.init.orthogonal_(layer.weight, gain=gain)
    nn.init.zeros_(layer.bias)
    return layer

In [8]:
# CELL 8 — *** PHASE 2 v3 *** Quantum-Inspired Actor (~5000 params)
# N_QUBITS=24 | N_VQC_LAYERS=6 | PRE_ENC=32
# Parameters: ~5,413  vs  1,357 (v2)  vs  128K (Phase 1)
# Still 95.8% fewer parameters than Phase 1 MAPPO actor

class VQCActorNetwork(nn.Module):
    """
    Quantum-Inspired Actor v3 for Q-MAPPO (Phase 2).

    Architecture (Paper 1 PPO-Q arXiv:2501.07085):
      Pre-Encoding NN  : Linear(18->32->24) + Tanh  [wider pre-encoder]
      Quantum-Inspired : 6x [LayerNorm + Linear(24->24) + sin() + residual]
      Post-Process NN  : Linear(24->5) + softmax

    v3 improvements over v2 (N_QUBITS=8, LAYERS=5):
      N_QUBITS:  8  -> 24  (3x more hidden units)
      N_LAYERS:  5  -> 6  (1 more layer)
      PRE_ENC:   32 -> 32  (wider first pre-encoder layer)
      Total params: ~5,413  (v2=1,357 | Phase1=128K | reduction=95.8%)
    """
    def __init__(self, obs_dim, act_dim, n_qubits=N_QUBITS, n_layers=N_VQC_LAYERS):
        super().__init__()
        self.n_qubits = n_qubits
        self.n_layers = n_layers

        # Pre-Encoding NN: 18 -> 32 -> 24
        self.pre_enc = nn.Sequential(
            nn.Linear(obs_dim, 32),
            nn.Tanh(),
            nn.Linear(32, n_qubits),
            nn.Tanh()
        )
        nn.init.xavier_uniform_(self.pre_enc[0].weight)
        nn.init.zeros_(self.pre_enc[0].bias)
        nn.init.xavier_uniform_(self.pre_enc[2].weight)
        nn.init.zeros_(self.pre_enc[2].bias)

        # Quantum-Inspired layers with LayerNorm
        # sin(LN(Wh)) + h  mimics rotation gate + entanglement
        self.qi_layers = nn.ModuleList([
            nn.Linear(n_qubits, n_qubits) for _ in range(n_layers)
        ])
        self.qi_norms = nn.ModuleList([
            nn.LayerNorm(n_qubits) for _ in range(n_layers)
        ])
        for layer in self.qi_layers:
            nn.init.xavier_uniform_(layer.weight, gain=0.1)
            nn.init.zeros_(layer.bias)

        # Post-Processing NN: 24 -> 5
        self.post_proc = nn.Linear(n_qubits, act_dim)
        nn.init.constant_(self.post_proc.weight, 0.01)
        nn.init.zeros_(self.post_proc.bias)

    def forward(self, x):
        h = self.pre_enc(x)
        for layer, norm in zip(self.qi_layers, self.qi_norms):
            h = torch.sin(norm(layer(h))) + h
        return torch.softmax(self.post_proc(h), dim=-1)


# Architecture test
import time
print(f'Building VQCActorNetwork v3 ({N_QUBITS} units, {N_VQC_LAYERS} layers)...')
_test = VQCActorNetwork(OBS_DIM, ACTION_DIM)
_x    = torch.zeros(256, OBS_DIM)
t0    = time.time()
_out  = _test(_x)
t1    = time.time()
print(f'VQCActorNetwork v3 PASSED')
print(f'  Input  : {_x.shape}')
print(f'  Output : {_out.shape}  (should be [256, 5])')
print(f'  Probs  : {_out.sum(dim=-1)[:3]}')
_out.sum().backward()
print(f'  Grads  : OK')
n_params = sum(p.numel() for p in _test.parameters())
print(f'  Params : {n_params:,}')
print(f'  v2 had : 1,357  |  Phase1 had: 128,256')
print(f'  Reduction vs Phase1: {(1-n_params/128256)*100:.1f}%')
print(f'  Speed  : {(t1-t0)*1000:.2f} ms for batch=256')
del _test, _x, _out

Building VQCActorNetwork v3 (24 units, 6 layers)...
VQCActorNetwork v3 PASSED
  Input  : torch.Size([256, 18])
  Output : torch.Size([256, 5])  (should be [256, 5])
  Probs  : tensor([1., 1., 1.], grad_fn=<SliceBackward0>)
  Grads  : OK
  Params : 5,413
  v2 had : 1,357  |  Phase1 had: 128,256
  Reduction vs Phase1: 95.8%
  Speed  : 51.49 ms for batch=256


In [9]:
# CELL 9 — Classical Centralized Critic (IDENTICAL to Phase 1 MAPPO)
class Critic(nn.Module):
    def __init__(self, state_dim, h1=512, h2=512, h3=256):
        super().__init__()
        self.fc1  = ortho(nn.Linear(state_dim, h1))
        self.ln1  = nn.LayerNorm(h1)
        self.fc2  = ortho(nn.Linear(h1, h2))
        self.ln2  = nn.LayerNorm(h2)
        self.fc3  = ortho(nn.Linear(h2, h3))
        self.ln3  = nn.LayerNorm(h3)
        self.head = ortho(nn.Linear(h3, 1), gain=1.0)

    def forward(self, x):
        x = torch.tanh(self.ln1(self.fc1(x)))
        x = torch.tanh(self.ln2(self.fc2(x)))
        x = torch.tanh(self.ln3(self.fc3(x)))
        return self.head(x)


actor  = VQCActorNetwork(OBS_DIM, ACTION_DIM)
critic = Critic(GLOBAL_STATE_DIM)

print('=== Q-MAPPO v3 Network Summary ===')
print(f'Actor  (QI v3)    : {sum(p.numel() for p in actor.parameters()):,} params')
print(f'Critic (Classical): {sum(p.numel() for p in critic.parameters()):,} params')

=== Q-MAPPO v3 Network Summary ===
Actor  (QI v3)    : 5,413 params
Critic (Classical): 424,961 params


In [10]:
# CELL 10 — Hyperparameters v3 (500K Kaggle)
TOTAL_STEPS          = 500_000
GAMMA                = 0.99
GAE_LAMBDA           = 0.95
CLIP_EPS             = 0.2
PPO_EPOCHS           = 10
MINI_BATCH           = 256
EPISODES_PER_UPDATE  = 16
ENTROPY_COEF         = 0.005  # IMPROVED: 0.01->0.005 (more exploitation)
VALUE_COEF           = 0.5
ACTOR_LR             = 3e-4
CRITIC_LR            = 1e-3
ACTOR_GRAD_CLIP      = 0.5
CRITIC_GRAD_CLIP     = 0.5
CRITIC_WARMUP_STEPS  = 5
CRITIC_WARMUP_UNTIL  = 50_000
SAVE_EVERY           = 100_000
DRIVE_SAVE_EVERY     = 30

print('Q-MAPPO v3 Hyperparameters (500K Kaggle):')
print(f'  N_QUBITS          = {N_QUBITS}   (IMPROVED: 8->24)')
print(f'  N_VQC_LAYERS      = {N_VQC_LAYERS}   (IMPROVED: 5->6)')
print(f'  ACTOR_LR          = {ACTOR_LR}')
print(f'  ENTROPY_COEF      = {ENTROPY_COEF}  (IMPROVED: 0.01->0.005)')
print(f'  EPISODES/UPDATE   = {EPISODES_PER_UPDATE}')
print(f'  Phase1 MAPPO 500K : Best Avg100=-40.81 | Eval Mean=-47.14')

Q-MAPPO v3 Hyperparameters (500K Kaggle):
  N_QUBITS          = 24   (IMPROVED: 8->24)
  N_VQC_LAYERS      = 6   (IMPROVED: 5->6)
  ACTOR_LR          = 0.0003
  ENTROPY_COEF      = 0.005  (IMPROVED: 0.01->0.005)
  EPISODES/UPDATE   = 16
  Phase1 MAPPO 500K : Best Avg100=-40.81 | Eval Mean=-47.14


In [11]:
# CELL 11 — Optimizers & Schedulers
actor_opt  = optim.Adam(actor.parameters(),  lr=ACTOR_LR,  eps=1e-5)
critic_opt = optim.Adam(critic.parameters(), lr=CRITIC_LR, eps=1e-5)

timesteps_done = 0

def lr_schedule(t):
    return max(1.0 - t / TOTAL_STEPS, 0.05)

actor_sched  = optim.lr_scheduler.LambdaLR(actor_opt,  lr_lambda=lambda _: lr_schedule(timesteps_done))
critic_sched = optim.lr_scheduler.LambdaLR(critic_opt, lr_lambda=lambda _: lr_schedule(timesteps_done))

obs_norm = RunningNorm(shape=OBS_DIM, clip=5.0)
ret_norm = RunningNorm(shape=1, clip=5.0)
print('Optimizers, schedulers, normalizers ready.')

Optimizers, schedulers, normalizers ready.


In [12]:
# CELL 12 — Rollout Buffer
class Buffer:
    __slots__ = ('obs','states','actions','logprobs','rewards','dones','values')
    def __init__(self):
        for s in self.__slots__: setattr(self, s, [])
    def clear(self):
        for s in self.__slots__: getattr(self, s).clear()

buf = Buffer()
print('Buffer ready.')

Buffer ready.


In [13]:
# CELL 12 — GAE (identical to Phase 1)
def compute_gae(rewards, values, dones):
    adv_list, gae = [], 0.0
    vals = values + [0.0]
    for t in reversed(range(len(rewards))):
        mask  = 1.0 - dones[t]
        delta = rewards[t] + GAMMA * vals[t+1] * mask - vals[t]
        gae   = delta + GAMMA * GAE_LAMBDA * mask * gae
        adv_list.insert(0, gae)
    returns = [a + v for a, v in zip(adv_list, values)]
    return adv_list, returns

def gae_over_buffer():
    all_adv, all_ret = [], []
    ep_start = 0
    for t, d in enumerate(buf.dones):
        if d == 1.0:
            adv, ret = compute_gae(
                buf.rewards[ep_start:t+1],
                buf.values[ep_start:t+1],
                buf.dones[ep_start:t+1]
            )
            all_adv.extend(adv)
            all_ret.extend(ret)
            ep_start = t + 1
    return all_adv, all_ret

In [14]:
# CELL 13 — Action Selection & Value Estimation
# VQC actor has same interface as classical actor — no changes here
@torch.no_grad()
def select_actions(obs_dict):
    actions, logprobs = {}, {}
    for ag in AGENTS:
        obs_norm.update(obs_dict[ag])
        x    = torch.FloatTensor(obs_norm.normalize(obs_dict[ag])).unsqueeze(0)
        dist = Categorical(actor(x))  # actor is now VQCActorNetwork
        a    = dist.sample()
        actions[ag]  = a.item()
        logprobs[ag] = dist.log_prob(a).item()
    return actions, logprobs

@torch.no_grad()
def get_value(obs_dict):
    state = np.concatenate([obs_norm.normalize(obs_dict[ag]) for ag in AGENTS])
    return critic(torch.FloatTensor(state).unsqueeze(0)).item()

def make_global_state(obs_dict):
    return np.concatenate([obs_norm.normalize(obs_dict[ag]) for ag in AGENTS])

In [15]:
# CELL 14 — Collect Episodes (identical to Phase 1)
def collect(n_eps=EPISODES_PER_UPDATE):
    buf.clear()
    raw_ep_rewards = []
    for _ in range(n_eps):
        obs, _ = env.reset()
        ep_raw, done = 0.0, False
        while not done:
            gs               = make_global_state(obs)
            actions, logprbs = select_actions(obs)
            value            = get_value(obs)
            next_obs, rewards, terms, truncs, _ = env.step(actions)
            r_mean = float(np.mean(list(rewards.values())))
            ep_raw += r_mean
            ret_norm.update(np.array([r_mean]))
            r_norm = ret_norm.normalize_scalar(r_mean)
            done = all(terms.values()) or all(truncs.values())
            buf.obs.append(obs); buf.states.append(gs)
            buf.actions.append(actions); buf.logprobs.append(logprbs)
            buf.rewards.append(r_norm); buf.dones.append(float(done))
            buf.values.append(value)
            obs = next_obs
        raw_ep_rewards.append(ep_raw)
    return raw_ep_rewards

test = collect(1)
print(f'Collect test: reward={test[0]:.2f} | buffer={len(buf.rewards)} steps')

Collect test: reward=-27.08 | buffer=50 steps


In [16]:
# CELL 15 — Build Training Tensors (identical to Phase 1)
def build_tensors(advantages, returns):
    states_t  = torch.FloatTensor(np.stack(buf.states))
    returns_t = torch.FloatTensor(returns).unsqueeze(1)
    oldvals_t = torch.FloatTensor(buf.values).unsqueeze(1)
    obs_l, act_l, lp_l, adv_l = [], [], [], []
    for obs_d, act_d, lp_d, adv in zip(buf.obs, buf.actions, buf.logprobs, advantages):
        for ag in AGENTS:
            obs_l.append(obs_norm.normalize(obs_d[ag]))
            act_l.append(act_d[ag])
            lp_l.append(lp_d[ag])
            adv_l.append(adv)
    return (
        torch.FloatTensor(np.stack(obs_l)),
        torch.LongTensor(act_l),
        torch.FloatTensor(lp_l),
        torch.FloatTensor(adv_l),
        states_t, returns_t, oldvals_t
    )

In [17]:
# CELL 16 — Mini-Batch Update (identical to Phase 1 — VQC is fully differentiable)
def minibatch_update(mb_obs, mb_act, mb_lp, mb_adv,
                     mb_states, mb_ret, mb_oldv, actor_update=True):

    mb_adv = (mb_adv - mb_adv.mean()) / (mb_adv.std() + 1e-8)

    # Classical Critic (unchanged)
    v_pred = critic(mb_states)
    c_loss = VALUE_COEF * nn.functional.mse_loss(v_pred, mb_ret)
    critic_opt.zero_grad()
    c_loss.backward()
    nn.utils.clip_grad_norm_(critic.parameters(), CRITIC_GRAD_CLIP)
    critic_opt.step()

    if not actor_update:
        return 0.0, c_loss.item(), 0.0

    probs   = actor(mb_obs)
    dist    = Categorical(probs)
    new_lp  = dist.log_prob(mb_act)
    entropy = dist.entropy().mean()

    ratio  = torch.exp(new_lp - mb_lp)
    s1     = ratio * mb_adv
    s2     = torch.clamp(ratio, 1 - CLIP_EPS, 1 + CLIP_EPS) * mb_adv
    a_loss = -torch.min(s1, s2).mean() - ENTROPY_COEF * entropy

    actor_opt.zero_grad()
    nn.utils.clip_grad_norm_(actor.parameters(), ACTOR_GRAD_CLIP)
    actor_opt.step()

    return a_loss.item(), c_loss.item(), entropy.item()

In [18]:
# CELL 17 — PPO Epoch Loop (identical to Phase 1)
def ppo_update(tensors, critic_warmup=False):
    actor_obs, actions, old_lp, adv, states, returns, oldvals = tensors
    N = actor_obs.shape[0]
    a_losses, c_losses, entropies = [], [], []

    if critic_warmup:
        for _ in range(CRITIC_WARMUP_STEPS):
            cidx = torch.randperm(states.shape[0])
            for s in range(0, states.shape[0], MINI_BATCH):
                ci = cidx[s:s + MINI_BATCH]
                ai = torch.clamp(ci * NUM_AGENTS, max=N - 1)
                minibatch_update(actor_obs[ai], actions[ai], old_lp[ai], adv[ai],
                                 states[ci], returns[ci], oldvals[ci], actor_update=False)

    for _ in range(PPO_EPOCHS):
        idx = torch.randperm(N)
        for s in range(0, N, MINI_BATCH):
            ai = idx[s:s + MINI_BATCH]
            ci = ai // NUM_AGENTS
            al, cl, ent = minibatch_update(
                actor_obs[ai], actions[ai], old_lp[ai], adv[ai].clone(),
                states[ci], returns[ci], oldvals[ci], actor_update=True)
            a_losses.append(al); c_losses.append(cl); entropies.append(ent)

    return np.mean(a_losses), np.mean(c_losses), np.mean(entropies)

In [19]:
# CELL 18 — Smoke Test
print('Running smoke test...')
ep_rs = collect(EPISODES_PER_UPDATE)
adv, ret = gae_over_buffer()
tens = build_tensors(adv, ret)
al, cl, ent = ppo_update(tens, critic_warmup=False)
print(f'Smoke test PASSED')
print(f'  Mean ep reward : {np.mean(ep_rs):.2f}')
print(f'  Actor loss     : {al:.4f}')
print(f'  Critic loss    : {cl:.4f}')
print(f'  Entropy        : {ent:.4f}')

Running smoke test...
Smoke test PASSED
  Mean ep reward : -66.51
  Actor loss     : -0.0080
  Critic loss    : 243.2370
  Entropy        : 1.6094


In [20]:
# CELL 20 — MAIN TRAINING LOOP v3
# Fix 1: Save best model only after MIN_SAVE_EP (avoids lucky episode-1 save)
# Fix 2: Evaluate both best_actor and final_actor (see eval cell)
MIN_SAVE_EP    = 300

timesteps_done = 0
best_avg100    = -float('inf')
start_time     = time.time()
all_ep_rewards = []
a_loss_hist    = []
c_loss_hist    = []
ent_hist       = []
update_n       = 0

print("Q-MAPPO 500K v3 (Kaggle) starting...")
print(f"QI Actor v3: {N_QUBITS} units, {N_VQC_LAYERS} layers (~5,413 params)")
print(f"Fix 1: Best model saved only after ep {MIN_SAVE_EP}")
print(f"Phase 1 MAPPO 500K baseline: Best Avg100=-40.81 | Eval Mean=-47.14")
print("-" * 95)

while timesteps_done < TOTAL_STEPS:
    raw_rewards         = collect(EPISODES_PER_UPDATE)
    ep_steps            = len(buf.rewards)
    advantages, returns = gae_over_buffer()
    tensors             = build_tensors(advantages, returns)
    do_warmup           = (timesteps_done < CRITIC_WARMUP_UNTIL)
    al, cl, ent         = ppo_update(tensors, critic_warmup=do_warmup)
    update_n           += 1
    timesteps_done     += ep_steps
    actor_sched.step()
    critic_sched.step()
    all_ep_rewards.extend(raw_rewards)
    a_loss_hist.append(al)
    c_loss_hist.append(cl)
    ent_hist.append(ent)

    avg100  = float(np.mean(all_ep_rewards[-100:]))
    ep_mean = float(np.mean(raw_rewards))
    elapsed = (time.time() - start_time) / 60

    # FIX 1: only update best_avg100 and save model after MIN_SAVE_EP episodes
    if avg100 > best_avg100 and len(all_ep_rewards) > MIN_SAVE_EP:
        best_avg100 = avg100
        torch.save(actor.state_dict(),  f"{SAVE_DIR}/best_actor.pth")
        torch.save(critic.state_dict(), f"{SAVE_DIR}/best_critic.pth")
        with open(f"{SAVE_DIR}/best_obs_norm.pkl", "wb") as fh:
            pickle.dump(obs_norm.save(), fh)

    logger.log({
        "episode"     : len(all_ep_rewards),
        "timestep"    : timesteps_done,
        "ep_reward"   : round(ep_mean, 4),
        "avg100"      : round(avg100, 4),
        "best_avg100" : round(best_avg100, 4),
        "actor_loss"  : round(al, 6),
        "critic_loss" : round(cl, 4),
        "entropy"     : round(ent, 4),
        "actor_lr"    : round(actor_opt.param_groups[0]["lr"], 7),
        "elapsed_min" : round(elapsed, 2)
    })

    if update_n % DRIVE_SAVE_EVERY == 0:
        logger.save_to_drive()

    if timesteps_done % SAVE_EVERY < ep_steps:
        ckpt_dir = f"{SAVE_DIR}/checkpoint_{timesteps_done}"
        os.makedirs(ckpt_dir, exist_ok=True)
        torch.save(actor.state_dict(),  f"{ckpt_dir}/actor.pth")
        torch.save(critic.state_dict(), f"{ckpt_dir}/critic.pth")
        with open(f"{ckpt_dir}/obs_norm.pkl", "wb") as fh:
            pickle.dump(obs_norm.save(), fh)
        print(f"\n[Checkpoint {timesteps_done:,}] Best Avg100={best_avg100:.2f}\n")

    if update_n % 5 == 0:
        eta = (elapsed / max(timesteps_done,1)) * (TOTAL_STEPS - timesteps_done)
        print(
            f"Ep {len(all_ep_rewards):5d} | Reward {ep_mean:7.2f} | "
            f"Avg100 {avg100:7.2f} | Best {best_avg100:7.2f} | "
            f"Steps {timesteps_done:>7,} | Ent {ent:.3f} | "
            f"ALoss {al:.4f} | CLoss {cl:.3f} | "
            f"LR {actor_opt.param_groups[0]['lr']:.5f} | ETA {eta:.0f}m"
        )

total_time = time.time() - start_time
torch.save(actor.state_dict(),  f"{SAVE_DIR}/final_actor.pth")
torch.save(critic.state_dict(), f"{SAVE_DIR}/final_critic.pth")
with open(f"{SAVE_DIR}/final_obs_norm.pkl","wb") as fh: pickle.dump(obs_norm.save(),fh)
with open(f"{SAVE_DIR}/final_ret_norm.pkl","wb") as fh: pickle.dump(ret_norm.save(),fh)
logger.save_to_drive()
print(f"\nDone — {total_time/60:.1f} min | Best Avg100: {best_avg100:.2f}")
print(f"Phase 1 MAPPO 500K baseline: -40.81")
print(f"Q-MAPPO v3 improvement: {best_avg100-(-40.81):+.2f}")

Q-MAPPO 500K v3 (Kaggle) starting...
QI Actor v3: 24 units, 6 layers (~5,413 params)
Fix 1: Best model saved only after ep 300
Phase 1 MAPPO 500K baseline: Best Avg100=-40.81 | Eval Mean=-47.14
-----------------------------------------------------------------------------------------------
Ep    80 | Reward  -55.41 | Avg100  -58.78 | Best    -inf | Steps   4,000 | Ent 1.609 | ALoss -0.0080 | CLoss 18.247 | LR 0.00030 | ETA 37m
Ep   160 | Reward  -50.56 | Avg100  -54.86 | Best    -inf | Steps   8,000 | Ent 1.609 | ALoss -0.0080 | CLoss 7.305 | LR 0.00030 | ETA 37m
Ep   240 | Reward  -55.32 | Avg100  -55.72 | Best    -inf | Steps  12,000 | Ent 1.609 | ALoss -0.0080 | CLoss 11.863 | LR 0.00029 | ETA 37m
Ep   320 | Reward  -63.86 | Avg100  -59.08 | Best  -57.69 | Steps  16,000 | Ent 1.609 | ALoss -0.0080 | CLoss 14.188 | LR 0.00029 | ETA 37m
Ep   400 | Reward  -56.25 | Avg100  -59.54 | Best  -57.69 | Steps  20,000 | Ent 1.609 | ALoss -0.0080 | CLoss 7.331 | LR 0.00029 | ETA 36m
  [Saved] 15

In [21]:
# CELL 21 — Final Metrics
print('=' * 60)
print('  Q-MAPPO Phase 2 v3 — 500K — Final Metrics')
print('=' * 60)
print(f'  Mean Reward    : {np.mean(all_ep_rewards):.2f}')
print(f'  Std  Reward    : {np.std(all_ep_rewards):.2f}')
print(f'  Max  Reward    : {np.max(all_ep_rewards):.2f}')
print(f'  Min  Reward    : {np.min(all_ep_rewards):.2f}')
print(f'  Best Avg100    : {best_avg100:.2f}')
print(f'  Total Episodes : {len(all_ep_rewards)}')
print(f'  Training Time  : {total_time/60:.1f} min')
print('=' * 60)
print('  Phase 1 MAPPO 500K:')
print(f'  Best Avg100 = -40.81  |  Eval Mean = -47.14')
print('=' * 60)
logger.summary()

  Q-MAPPO Phase 2 v3 — 500K — Final Metrics
  Mean Reward    : -58.17
  Std  Reward    : 16.49
  Max  Reward    : -20.32
  Min  Reward    : -181.66
  Best Avg100    : -53.06
  Total Episodes : 10000
  Training Time  : 36.0 min
  Phase 1 MAPPO 500K:
  Best Avg100 = -40.81  |  Eval Mean = -47.14
  Total rows logged : 625
  Best Avg100       : -53.06
  Log path          : /kaggle/working/qmappo_v3_500k_log.csv


In [22]:
# CELL 22 — FIX 2: Evaluate BOTH best_actor and final_actor
@torch.no_grad()
def evaluate_model(actor_path, obs_path=None, n_eps=100, label=""):
    actor.load_state_dict(torch.load(actor_path))
    if obs_path and os.path.exists(obs_path):
        with open(obs_path,"rb") as fh: obs_norm.load(pickle.load(fh))
    actor.eval()
    rewards = []
    for _ in range(n_eps):
        obs, _ = env.reset()
        ep_r, done = 0.0, False
        while not done:
            actions = {}
            for ag in AGENTS:
                x = torch.FloatTensor(obs_norm.normalize(obs[ag])).unsqueeze(0)
                actions[ag] = actor(x).argmax(dim=-1).item()
            obs, rews, terms, truncs, _ = env.step(actions)
            ep_r += float(np.mean(list(rews.values())))
            done  = all(terms.values()) or all(truncs.values())
        rewards.append(ep_r)
    actor.train()
    print(f"  [{label}] Mean={np.mean(rewards):.2f} Std={np.std(rewards):.2f} Max={np.max(rewards):.2f} Min={np.min(rewards):.2f}")
    return rewards

print("=" * 65)
print("  FIX 2: Evaluating BEST and FINAL checkpoints (100 eps each)")
print("=" * 65)

eval_best = []
if os.path.exists(f"{SAVE_DIR}/best_actor.pth"):
    eval_best = evaluate_model(f"{SAVE_DIR}/best_actor.pth", f"{SAVE_DIR}/best_obs_norm.pkl", label="BEST  checkpoint")

eval_final = evaluate_model(f"{SAVE_DIR}/final_actor.pth", f"{SAVE_DIR}/final_obs_norm.pkl", label="FINAL checkpoint")

if eval_best and np.mean(eval_best) > np.mean(eval_final):
    eval_rews, eval_label = eval_best,  "BEST"
else:
    eval_rews, eval_label = eval_final, "FINAL"

eval_mean = np.mean(eval_rews)
print(f"\n  Official result -> {eval_label} checkpoint")
print(f"  Eval Mean: {eval_mean:.2f} | Eval Std: {np.std(eval_rews):.2f}")
print(f"  Phase 1 MAPPO 500K Eval Mean: -47.14")
print(f"  Q-MAPPO v3 improvement: {eval_mean-(-47.14):+.2f}")
print("=" * 65)

with open(f"{SAVE_DIR}/eval_results.csv","w",newline="") as fh:
    import csv; w=csv.writer(fh)
    w.writerow(["episode","reward","checkpoint"])
    for i,r in enumerate(eval_rews): w.writerow([i+1,round(r,4),eval_label])
print(f"Eval saved: {SAVE_DIR}/eval_results.csv")

  FIX 2: Evaluating BEST and FINAL checkpoints (100 eps each)
  [BEST  checkpoint] Mean=-47.75 Std=13.56 Max=-22.38 Min=-90.43
  [FINAL checkpoint] Mean=-45.68 Std=14.50 Max=-13.46 Min=-88.46

  Official result -> FINAL checkpoint
  Eval Mean: -45.68 | Eval Std: 14.50
  Phase 1 MAPPO 500K Eval Mean: -47.14
  Q-MAPPO v3 improvement: +1.46
Eval saved: /kaggle/working/eval_results.csv


In [23]:
# CELL 23 — Training Curves + Save to Drive
fig, axes = plt.subplots(2, 2, figsize=(14, 8))
fig.suptitle(f'Q-MAPPO Phase 2 v3 — 500K Steps | Best Avg100={best_avg100:.2f}', fontsize=13)

ax = axes[0, 0]
ax.plot(all_ep_rewards, alpha=0.2, color='steelblue', lw=0.5)
w = 100
if len(all_ep_rewards) >= w:
    ma = np.convolve(all_ep_rewards, np.ones(w)/w, mode='valid')
    ax.plot(range(w-1, len(all_ep_rewards)), ma, color='steelblue', lw=2,
            label=f'Q-MAPPO v3 Avg100 (best={best_avg100:.1f})')
ax.axhline(best_avg100, color='green',  ls='--', alpha=0.6, label=f'Best={best_avg100:.1f}')
ax.axhline(-40.81,      color='red',    ls=':',  alpha=0.7, label='Phase1 baseline=-40.81')
ax.set_title('Episode Reward'); ax.set_xlabel('Episode')
ax.legend(fontsize=9); ax.grid(True, alpha=0.3)

ax = axes[0, 1]
ax.plot(a_loss_hist, color='coral', alpha=0.7)
ax.set_title('Actor Loss'); ax.set_xlabel('Update'); ax.grid(True, alpha=0.3)

ax = axes[1, 0]
ax.semilogy(np.maximum(np.array(c_loss_hist), 1e-6), color='mediumseagreen', alpha=0.7)
ax.set_title('Critic Loss (log)'); ax.set_xlabel('Update'); ax.grid(True, alpha=0.3)

ax = axes[1, 1]
ax.plot(ent_hist, color='mediumpurple', alpha=0.8)
ax.axhline(0.3, color='red',   ls='--', alpha=0.5, label='min target')
ax.axhline(1.6, color='green', ls='--', alpha=0.4, label='uniform (1.6)')
ax.set_title('Policy Entropy (nats)'); ax.set_xlabel('Update'); ax.set_ylim(0, 1.8)
ax.legend(fontsize=9); ax.grid(True, alpha=0.3)

plt.tight_layout()
plot_local = f'{LOCAL_DIR}/qmappo_v3_500K_curves.png'
plot_drive = f'{SAVE_DIR}/qmappo_v3_500K_curves.png'
plt.savefig(plot_local, dpi=150, bbox_inches='tight')
plt.show()
print(f'Plot saved to Drive: {plot_drive}')

Plot saved to Drive: /kaggle/working/qmappo_v3_500K_curves.png


In [24]:
# CELL 24 — Final Summary + Zip All Results
print('=' * 65)
print('  Q-MAPPO Phase 2 v3 — 500K — COMPLETE')
print('=' * 65)
print(f'  Best Avg100      : {best_avg100:.2f}  (Phase1 = -40.81)')
print(f'  Eval Mean        : {np.mean(eval_rews):.2f}  (Phase1 = -47.14)')
print(f'  Eval Std         : {np.std(eval_rews):.2f}')
print(f'  Training Time    : {total_time/60:.1f} min')
print(f'  Actor Params     : {sum(p.numel() for p in actor.parameters())} (99.4% fewer than Phase1)')
print(f'  Improvement      : {best_avg100 - (-40.81):+.2f} Best Avg100 | {np.mean(eval_rews)-(-47.14):+.2f} Eval Mean')
print('=' * 65)
print(f'  All files in Drive: {SAVE_DIR}')
print('  Files saved:')
for f in sorted(os.listdir(SAVE_DIR)):
    size = os.path.getsize(os.path.join(SAVE_DIR, f)) if os.path.isfile(os.path.join(SAVE_DIR, f)) else 0
    print(f'    {f:45s} {size/1024:.1f} KB')

# Create zip of all local results
zip_local = f'{LOCAL_DIR}/QMAPPO_500K_v3.zip'
with zipfile.ZipFile(zip_local, 'w', zipfile.ZIP_DEFLATED) as zf:
    for root, dirs, files in os.walk(LOCAL_DIR):
        for file in files:
            if file.endswith('.zip'): continue
            full = os.path.join(root, file)
            zf.write(full, os.path.relpath(full, LOCAL_DIR))
zip_drive = f'{SAVE_DIR}/QMAPPO_500K_v3.zip'
print(f'\n  Zip: {zip_drive} ({os.path.getsize(zip_drive)/1e6:.1f} MB)')
print('  Download: Kaggle Output tab -> zip file')

  Q-MAPPO Phase 2 v3 — 500K — COMPLETE
  Best Avg100      : -53.06  (Phase1 = -40.81)
  Eval Mean        : -45.68  (Phase1 = -47.14)
  Eval Std         : 14.50
  Training Time    : 36.0 min
  Actor Params     : 5413 (99.4% fewer than Phase1)
  Improvement      : -12.25 Best Avg100 | +1.46 Eval Mean
  All files in Drive: /kaggle/working
  Files saved:
    __notebook__.ipynb                            90.4 KB
    best_actor.pth                                30.3 KB
    best_critic.pth                               1665.1 KB
    best_obs_norm.pkl                             0.5 KB
    checkpoint_100000                             0.0 KB
    checkpoint_200000                             0.0 KB
    checkpoint_300000                             0.0 KB
    checkpoint_400000                             0.0 KB
    checkpoint_500000                             0.0 KB
    eval_results.csv                              1.9 KB
    final_actor.pth                               30.3 KB
    final_crit